# Database-Level Multi-Tenancy in Milvus

This notebook demonstrates how to implement database-level isolation for multiple tenants using Milvus. Each tenant gets their own dedicated database, providing the strongest form of tenant isolation.

## Prerequisites
- Running Milvus instance (v2.5.x or later)
- PyMilvus SDK (v2.5.8 or compatible)

## Setup and Configuration

In [ ]:
# Install required packages if not already installed
# !pip install pymilvus==2.5.8

In [ ]:
from pymilvus import MilvusClient, DataType
import random

# Configuration
MILVUS_URI = "http://localhost:19530"  # Update this to your Milvus instance
client = MilvusClient(uri=MILVUS_URI)

print(f"Connected to Milvus at {MILVUS_URI}")

## Define Tenant Configurations

Each tenant will have their own database with specific collections.

In [ ]:
# Define tenant configurations
TENANTS = {
    "company_a": {"db_name": "tenant_company_a", "collections": ["products", "users"]},
    "company_b": {
        "db_name": "tenant_company_b",
        "collections": ["inventory", "customers"],
    },
    "company_c": {
        "db_name": "tenant_company_c",
        "collections": ["documents", "analytics"],
    },
}

print(f"Configured {len(TENANTS)} tenants:")
for tenant_id, config in TENANTS.items():
    print(
        f"  - {tenant_id}: {config['db_name']} with collections {config['collections']}"
    )

## Tenant Database Setup Functions

In [ ]:
def setup_tenant_database(tenant_id, config):
    """Set up a dedicated database for a tenant"""
    db_name = config["db_name"]

    # Create database for tenant
    try:
        client.create_database(db_name)
        print(f"✓ Created database '{db_name}' for {tenant_id}")
    except Exception as e:
        print(f"Database '{db_name}' may already exist: {e}")

    # Switch to tenant database
    client.using_database(db_name)

    # Create collections for tenant
    for coll_name in config["collections"]:
        schema = client.create_schema(auto_id=True, enable_dynamic_fields=True)

        # Add fields (example schema)
        schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
        schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=768)
        schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=1000)
        schema.add_field(
            field_name="category", datatype=DataType.VARCHAR, max_length=100
        )

        # Create collection
        try:
            client.create_collection(collection_name=coll_name, schema=schema)
            print(f"✓ Created collection '{coll_name}' in '{db_name}'")

            # Create index for vector field
            client.create_index(
                collection_name=coll_name,
                field_name="vector",
                index_params={"index_type": "FLAT", "metric_type": "COSINE"},
            )
            print(f"✓ Created index for '{coll_name}'")

        except Exception as e:
            print(f"Collection '{coll_name}' may already exist: {e}")

    # Switch back to default database
    client.using_database("default")
    return True

In [ ]:
def insert_sample_data(tenant_id, collection_name, num_records=100):
    """Insert sample data into a tenant's collection"""
    db_name = TENANTS[tenant_id]["db_name"]

    # Switch to tenant database
    client.using_database(db_name)

    # Generate sample data
    data = []
    categories = ["electronics", "books", "clothing", "home", "sports"]

    for i in range(num_records):
        data.append(
            {
                "vector": [random.random() for _ in range(768)],
                "text": f"{tenant_id} item {i + 1}",
                "category": random.choice(categories),
            }
        )

    # Insert data
    client.insert(collection_name=collection_name, data=data)

    # Load collection for searching
    client.load_collection(collection_name)

    print(f"✓ Inserted {num_records} records into {tenant_id}/{collection_name}")

    # Switch back to default database
    client.using_database("default")
    return True

In [ ]:
def query_tenant_data(tenant_id, collection_name, query_vector, limit=5):
    """Query data from a specific tenant's database"""
    db_name = TENANTS[tenant_id]["db_name"]

    # Switch to tenant database
    client.using_database(db_name)

    # Perform search - data is completely isolated
    results = client.search(
        collection_name=collection_name,
        data=[query_vector],
        limit=limit,
        output_fields=["text", "category"],
    )

    # Switch back to default database
    client.using_database("default")

    return results

## Setup All Tenant Databases

Create separate databases for each tenant with their specific collections.

In [ ]:
# Setup databases for all tenants
print("Setting up tenant databases...")
for tenant_id, config in TENANTS.items():
    print(f"\nSetting up {tenant_id}:")
    setup_tenant_database(tenant_id, config)

## Insert Sample Data for Each Tenant

In [ ]:
# Insert sample data for each tenant
print("Inserting sample data...")
for tenant_id, config in TENANTS.items():
    print(f"\nInserting data for {tenant_id}:")
    for collection_name in config["collections"]:
        insert_sample_data(tenant_id, collection_name, num_records=50)

## Demonstrate Tenant Isolation

Show that each tenant can only access their own data.

In [ ]:
# Generate a query vector
query_vector = [random.random() for _ in range(768)]

print("Demonstrating tenant isolation:")
print("\nQuerying Company A's products:")
results_a = query_tenant_data("company_a", "products", query_vector, limit=3)
for i, result in enumerate(results_a[0]):
    print(
        f"  {i + 1}. {result['entity']['text']} ({result['entity']['category']}) - Score: {result['distance']:.4f}"
    )

print("\nQuerying Company B's inventory:")
results_b = query_tenant_data("company_b", "inventory", query_vector, limit=3)
for i, result in enumerate(results_b[0]):
    print(
        f"  {i + 1}. {result['entity']['text']} ({result['entity']['category']}) - Score: {result['distance']:.4f}"
    )

print("\nQuerying Company C's documents:")
results_c = query_tenant_data("company_c", "documents", query_vector, limit=3)
for i, result in enumerate(results_c[0]):
    print(
        f"  {i + 1}. {result['entity']['text']} ({result['entity']['category']}) - Score: {result['distance']:.4f}"
    )

## Verify Database Isolation

In [ ]:
# List all databases to verify isolation
databases = client.list_databases()
print("All databases in Milvus:")
for db in databases:
    print(f"  - {db}")

print(
    f"\nTotal tenant databases created: {len([db for db in databases if db.startswith('tenant_')])}"
)

## Test Cross-Tenant Access (Should Fail)

Demonstrate that tenants cannot access each other's data.

In [ ]:
# Try to access Company B's data from Company A's database context
print("Testing cross-tenant access restriction:")
try:
    # Switch to Company A's database
    client.using_database(TENANTS["company_a"]["db_name"])

    # Try to access Company B's collection (this should fail)
    results = client.search(
        collection_name="inventory",  # Company B's collection
        data=[query_vector],
        limit=5,
    )
    print("❌ Cross-tenant access succeeded (this shouldn't happen!)")
except Exception as e:
    print(f"✓ Cross-tenant access properly blocked: {str(e)[:100]}...")
finally:
    # Switch back to default database
    client.using_database("default")

## Performance and Resource Analysis

In [ ]:
# Analyze resource usage per tenant
print("Database-Level Multi-Tenancy Analysis:")
print("\nBenefits:")
print("  ✓ Maximum data isolation (100% secure)")
print("  ✓ Simple tenant lifecycle management")
print("  ✓ Granular RBAC control per database")
print("  ✓ Zero risk of data leakage between tenants")

print("\nLimitations:")
print("  ⚠ Limited to ~64 tenants (Milvus database limit)")
print("  ⚠ 40-60% higher memory usage per tenant")
print("  ⚠ 3-5x more operational complexity")
print("  ⚠ Higher infrastructure costs")

total_collections = sum(len(config["collections"]) for config in TENANTS.values())
print("\nCurrent Setup:")
print(f"  - Total tenants: {len(TENANTS)}")
print(f"  - Total databases: {len(TENANTS)}")
print(f"  - Total collections: {total_collections}")
print(f"  - Average collections per tenant: {total_collections / len(TENANTS):.1f}")

## Cleanup (Optional)

Uncomment and run the following cell to clean up the created databases.

In [ ]:
# Cleanup - Uncomment to remove all tenant databases
# print("Cleaning up tenant databases...")
# for tenant_id, config in TENANTS.items():
#     try:
#         client.drop_database(config["db_name"])
#         print(f"✓ Dropped database {config['db_name']}")
#     except Exception as e:
#         print(f"Error dropping {config['db_name']}: {e}")
# print("Cleanup completed!")

## Summary

This notebook demonstrated database-level multi-tenancy in Milvus, which provides:

- **Strongest Isolation**: Complete data separation at the database level
- **Simple Management**: Each tenant has their own namespace
- **Security**: Zero risk of cross-tenant data access
- **Scalability Limit**: Suitable for up to 64 enterprise tenants

This approach is ideal for enterprise customers requiring strict compliance and data isolation guarantees.